# Data Preparation Template

## This notebook is for Data Cleaning and Feature Engineering

**==============================================================================================================**

### Markdown Guides

> This is a blockquote.

Some of these words *are emphasized*.

Use two asterisks for **strong emphasis**.

*   Another item in the list.

This is an [example link](http://example.com/).

$x = x + y$

[text to appear as link](#linkhandle)

Images inline
![image](6-step-ml-framework.png)

## Data Dictionary

| Field          | Description                                                                           |
|----------------|---------------------------------------------------------------------------------------|
| |	|
| |	|
| |	|
| |	|
| |	|
| |	|
| |	|
| |	|
| |	|
| |	|
| |	|
| |	|
| |	|
| |	|

## Data Tasks

### 1) Understand the shape of the data (Histograms, box plots, etc.)

### 2) Data Cleaning 

### 3) Data Exploration

### 4) Feature Engineering 

### 5) Data Preprocessing for Model

### 6) Basic Model Building 

### 7) Model Tuning 

### 8) Ensemble Model Building 

### 9) Results 


**==============================================================================================================**

## Import Libraries

In [ ]:
import numpy as np
#from numpy import count_nonzero, median, mean
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
#import squarify

import datetime
from datetime import datetime, timedelta, date, time


#import os
#import zipfile
import scipy
from scipy import stats
#from scipy.stats.mstats import normaltest # D'Agostino K^2 Test
#from scipy.stats import boxcox
from collections import Counter

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, Binarizer
from sklearn.impute import SimpleImputer, MissingIndicator, KNNImputer
from sklearn.compose import make_column_transformer, ColumnTransformer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn import set_config

%matplotlib inline
#sets the default autosave frequency in seconds
%autosave 60 
sns.set_style('dark')
sns.set(font_scale=1.2)

plt.rc('axes', titlesize=9)
plt.rc('axes', labelsize=14)
plt.rc('xtick', labelsize=12)
plt.rc('ytick', labelsize=12)

import warnings
warnings.filterwarnings('ignore')

# Use Feature-Engine library
import feature_engine

from feature_engine.imputation import AddMissingIndicator, CategoricalImputer, DropMissingData, MeanMedianImputer
from feature_engine.imputation import ArbitraryNumberImputer, RandomSampleImputer

from feature_engine.outliers import Winsorizer, ArbitraryOutlierCapper, OutlierTrimmer

from feature_engine.encoding import CountFrequencyEncoder, DecisionTreeEncoder, MeanEncoder, OneHotEncoder
from feature_engine.encoding import OrdinalEncoder, WoEEncoder, RareLabelEncoder, StringSimilarityEncoder

from feature_engine.discretisation import EqualWidthDiscretiser, EqualFrequencyDiscretiser, ArbitraryDiscretiser
from feature_engine.discretisation import DecisionTreeDiscretiser, EqualWidthDiscretiser

from feature_engine.datetime import DatetimeFeatures

from feature_engine.creation import CyclicalFeatures, MathFeatures, RelativeFeatures


pd.set_option('display.max_columns',None)
#pd.set_option('display.max_rows',None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format','{:.2f}'.format)

random.seed(0)
np.random.seed(0)
np.set_printoptions(suppress=True)

**==============================================================================================================**

## Data Quick Glance

In [ ]:
df = pd.read_csv("cyclistic.csv")

In [ ]:
#df = pd.read_csv("bikeshare.csv", parse_dates=['startedate'])

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.dtypes.value_counts()

In [ ]:
# Descriptive Statistical Analysis
df.describe(include="all")

In [ ]:
# Descriptive Statistical Analysis
df.describe(include=["int", "float"])

In [ ]:
# Descriptive Statistical Analysis
df.describe(include="object")

In [ ]:
df.columns

In [ ]:
df.zipcodestart.value_counts(dropna=False)

In [ ]:
df.zipcodestart.nunique()

In [ ]:
df.boroughstart.value_counts()

In [ ]:
df.neighborhoodstart.value_counts()

In [ ]:
df.neighborhoodend.value_counts()

In [ ]:
# Check target variable

df.usertype.value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

**==============================================================================================================**

## Pivot Tables

In [ ]:
df2 = pd.pivot_table(data=df, index=['default'], aggfunc='median')
df2

In [ ]:
df2.reset_index(inplace=True)

In [ ]:
df2

In [ ]:
df2.drop(["id","year"], axis=1, inplace=True)

In [ ]:
df2

In [ ]:
#df2.to_csv("airfare.csv", index=False)

**==============================================================================================================**

## SciKit Learn Column Transformers and Pipelines

In [ ]:
list(df.select_dtypes(["int", "float"]))

In [ ]:
list(df.select_dtypes(["object"]))

In [ ]:
numcols = ['RankSeason', 'RankPlayoffs','OOBP', 'OSLG']

In [ ]:
catcols = ['windgustdir', 'winddir9am', 'winddir3pm']

In [ ]:
df.shape

In [ ]:
X = df.iloc[:, 0:15]
y = df.iloc[:, 15:]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# imp = SimpleImputer()
# ss = StandardScaler()
# mm = MinMaxScaler()
# ohe = OneHotEncoder(drop_last=True)
# binary = Binarizer(threshold=0)

## You use the ColumnTransformer to transform each column set separately before combining them later.

In [ ]:
# Missing Values Imputation Methods

imp = SimpleImputer(missing_values=np.nan,
                    strategy='mean',
                    fill_value=None,
                    add_indicator=False)

imp = SimpleImputer(missing_values=np.nan,
                    strategy='median',
                    fill_value=None,
                    add_indicator=False)

imp = SimpleImputer(missing_values=np.nan,
                    strategy='constant',
                    fill_value=0,
                    add_indicator=False)

imp = SimpleImputer(missing_values=np.nan,
                    strategy='most_frequent',
                    fill_value=None,
                    add_indicator=False)

indicator = MissingIndicator(missing_values=np.nan, features='missing-only', error_on_new=True)
indicator

imp2 = SimpleImputer(missing_values=np.nan,
                    strategy='most_frequent',
                    fill_value=None,
                    add_indicator=True)

In [ ]:
# Missing values imputation using ColumnTransformer

ct      = ColumnTransformer(
    
          transformers= [
     
         (
             'imputer', SimpleImputer(missing_values=np.nan, 
                                      strategy='most_frequent', add_indicator=True), ['RankSeason', 'RankPlayoffs','OOBP', 'OSLG']
         )
         
         ],
             remainder='passthrough',
             verbose_feature_names_out=False
         )

ct.set_output(transform="pandas")

**CAUTION**

The ColumnTransformer is, in essence, just slicing the dataframe into the required feature subsets. The SimpleImputer then performs operations on the sliced dataframes. Finally, the dataframes are put back together for the final output.

That means that the order of the columns is not the same as in the training set!

In [ ]:
# One hot / Ordinal encoding
# With a tree-based model, try OrdinalEncoder instead of OneHotEncoder even for nominal (unordered) features

ct = ColumnTransformer(
    
     transformers= [
     
     (
         'ohe1', OneHotEncoder(drop_last_binary=True), []
     ),  
         
     (
         'ohe2', OneHotEncoder(top_categories=3), []
     ),
         
     ( 
         'oe',  OrdinalEncoder(missing_values='ignore'), ['windgustdir', 'winddir9am', 'winddir3pm']
     ),
         
     (   'binary', Binarizer(threshold=0),[]
     
     )
         
     ],
      remainder='passthrough',
      verbose_feature_names_out=False
     )

ct.set_output(transform="pandas")

In [ ]:
# Scaling features

ct = ColumnTransformer(
    
     transformers= [
     
      (
         'ss', 
         StandardScaler(),
         ['crim', 'zn', 'indus',  'nox', 'rm', 'age', 'dis', 'rad', 'tax', 'ptratio', 'black', 'lstat']
      ),    
      ( 
          'mm',
          MinMaxScaler(),
          ['campaign', 'pdays', 'previous']
      )
         
     ],
      remainder='passthrough',
      verbose_feature_names_out=False
     )

ct.set_output(transform="pandas")

In [ ]:
X_new = ct.fit_transform(X,y)
X_new

In [ ]:
X_new.isnull().sum()

In [ ]:
print(ct)

In [ ]:
df2 = X_new.copy()
df2.head()

## Use the pipeline for multiple transformations of the same columns

### Step 1: Create pipelines for numerical and categorical features

```
pipe = Pipeline(steps, *, memory=None, verbose=False)

We create the preprocessing pipelines for both numerical and categorical data

```

In [ ]:
numpipeline = Pipeline(steps=
                      [
                      ("imputer", SimpleImputer()),
                      ("scaler", StandardScaler())    
                      ]
                      )

In [ ]:
catpipeline = Pipeline(steps=
                      [
                      ("imputer", SimpleImputer()),    
                      ("ohe", OneHotEncoder())    
                      ]
                      )

### Step 2: Create Transformers

In [ ]:
preprocessor = ColumnTransformer(
    
     transformers= [
     
     ("numerical", numpipeline, numcols ),    
     ("categorical", catpipeline, catcols )
         
     ],
      remainder='passthrough',
      verbose_feature_names_out=True
     )

preprocessor.set_output(transform="pandas")

### Step 3: Create a final pipeline to include transformers

In [ ]:
finalpipeline = Pipeline(steps=
                       [
                       ("preprocessor", preprocessor)
                       ])

In [ ]:
finalpipeline.fit_transform(X)

In [ ]:
finalpipeline.named_steps

**==============================================================================================================**

## Overall Visualization

In [ ]:
df.hist(bins=50, figsize=(20,50), layout=(len(df.columns),2), grid=False)
plt.suptitle('Histogram Feature Distribution', x=0.5, y=1.02, ha='center', fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
df.boxplot(figsize=(20,10), color='blue', fontsize=15)
plt.suptitle('BoxPlots Feature Distribution', x=0.5, y=1.02, ha='center', fontsize=20)

plt.tight_layout()
plt.show()

In [ ]:
# Stacked Histogram

fig, ax = plt.subplots(figsize=(12,8))

sns.histplot(data=df, x="landsurfacecondition", y=None, hue="damagegrade", multiple='dodge', stat='count')

plt.show()

In [ ]:
# Stacked Histogram

fig, ax = plt.subplots(figsize=(12,8))

sns.histplot(data=df, x="foundationtype", y=None, hue="damagegrade", multiple='dodge', stat='count')

plt.show()

In [ ]:
# Stacked Histogram

fig, ax = plt.subplots(figsize=(12,8))

sns.histplot(data=df, x="rooftype", y=None, hue="damagegrade", multiple='dodge', stat='count')

plt.show()

In [ ]:
# Stacked Histogram

fig, ax = plt.subplots(figsize=(12,8))

sns.histplot(data=df, x="groundfloortype", y=None, hue="damagegrade", multiple='dodge', stat='count')

plt.show()

In [ ]:
# Stacked Histogram

fig, ax = plt.subplots(figsize=(12,8))

sns.histplot(data=df, x="position", y=None, hue="damagegrade", multiple='dodge', stat='count')

plt.show()

In [ ]:
# Stacked Histogram

fig, ax = plt.subplots(figsize=(12,8))

sns.histplot(data=df, x="planconfiguration", y=None, hue="damagegrade", multiple='dodge', stat='count')

plt.show()

In [ ]:
# Stacked Histogram

fig, ax = plt.subplots(figsize=(12,8))

sns.histplot(data=df, x="legalownershipstatus", y=None, hue="damagegrade", multiple='dodge', stat='count')

plt.show()

In [ ]:
sns.catplot(x="landsurfacecondition", y="damagegrade", hue=None, data=df, row=None, col=None,
    col_wrap=None, estimator="mean", ci=95, kind='swarm', height=5, aspect=3)

plt.show()

In [ ]:
sns.barplot(x="label", y="drivenkmdrives", hue=None, data=df, estimator="mean", ci=95)

plt.show()

In [ ]:
sns.barplot(x=horizontal_label,
            y=first_dimension,
            hue=second_dimension,
            data=df.groupby([first_dimension, second_dimension]).size().to_frame(horizontal_label).reset_index())

In [ ]:
sns.pairplot(data=df, 
             height=5, aspect=1, kind="reg",
             #x_vars=['extra','mtatax','tollsamount','improvementsurcharge'],
             y_vars=["calories"]
            
            )

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))

sns.scatterplot(x="NumberChildrenAtHome", y="YearlyIncome", hue="BikeBuyer", data=df, 
            size="Gender", ci=95)

plt.show()

In [ ]:
sns.relplot(x="NumberChildrenAtHome", y="YearlyIncome", hue="BikeBuyer", data=df, kind='scatter',
            row=None, col=None, col_wrap=None, height=4, aspect=3)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))

sns.lineplot(x="NumberChildrenAtHome", y="NumberCarsOwned", hue="HomeOwnerFlag", size=None, style=None, data=df,
            estimator='mean', ci=95)

plt.show()

In [ ]:
sns.catplot(x="CountryRegionName", y="NumberCarsOwned", hue=None, data=df, row=None, col=None,
    col_wrap=None, estimator="mean", ci=95, kind='strip', height=4, aspect=3)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))

sns.countplot(x="verifiedstatus", y=None, hue=None, data=df)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))

sns.violinplot(x="age", y=None, hue=None, data=df, split=True)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))
sns.kdeplot(x="income", y=None, shade=True, vertical=False, kernel=None, data=df)

plt.show()

In [ ]:
sns.lmplot(x=None, y=None, data=None, hue=None, col=None, row=None, col_wrap=None,
    height=5, aspect=1)

plt.show()

In [ ]:
sns.jointplot(x=None, y=None, data=None, kind='scatter', color=None, height=6, ratio=5, hue=None)

plt.show()

In [ ]:
sns.jointplot(x="landsurfacecondition", y="damagegrade", data=df, kind='hist', color=None, height=6, ratio=5, hue=None)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))


plt.show()

In [ ]:
g = sns.FacetGrid(data=df, row=None, col="damagegrade", hue=None, col_wrap=None,  sharex=True,  sharey=True,
              height=5, aspect=1)

g.map(sns.histplot, "countfloorspreeq")

plt.show()

In [ ]:
g = sns.FacetGrid(data=df, row=None, col="damagegrade", hue=None, col_wrap=None,  sharex=True,  sharey=True,
              height=5, aspect=1)

g.map(sns.histplot, "area")

plt.show()

**==============================================================================================================**

## Data Replacement

In [ ]:
list(df.select_dtypes("object"))

In [ ]:
df["poutcome"].value_counts()

In [ ]:
df["poutcome"] = df["poutcome"].replace(to_replace="other", value="failure")

In [ ]:
df["poutcome"].value_counts()

In [ ]:
df["month"].value_counts()

In [ ]:
df["contact"] = df["contact"].replace(to_replace="unknown", value="telephone")

In [ ]:
df["contact"].value_counts()

In [ ]:
#df.to_csv("bank.csv", index=False)

**==============================================================================================================**

## Data Filtering

### Filtering with logical operators

We can use the logical operators on column values to filter rows. First, we  specify the name of our data, then, square brackets to select the name of the column, double 'equal' sign, '==' to select the name of a row group, in single or double quotation marks. If we want to exclude some entries (e.g. some locations), we would use the 'equal' and 'exclamation point' signs together, '=!'. We can also use '</>', '<=/>=' signs to select numeric information.

```
FBJan = df[df['FBAdCampaign'] == 'FB_Jan19']
FBJan.head()
```

In [ ]:
df.columns

In [ ]:
df['nativecountry'].value_counts()

In [ ]:
df2 = df[df['nativecountry'] == ' United-States']
df2

In [ ]:
df2

In [ ]:
df2.reset_index(drop=True, inplace=True)

In [ ]:
df2

In [ ]:
df2.duration.describe()

In [ ]:
#df2.to_csv("nyctaximod.csv", index=False)

### Filtering by multiple conditions

There are many alternative ways to perform filtering in pandas. We can also use '|' ('or') and '&' (and) to select multiple columns and rows. 

In [ ]:
mult_loc = df[(df['FBAdCampaign'] == "FB_Jan19") | (df['AdWordsAdCampaign'] == "AW_Dec19")]
mult_loc

In [ ]:
cities = ['Calgary', 'Toronto', 'Edmonton']
CTE = data[data.City.isin(cities)]
CTE

**==============================================================================================================**

# Data Preprocessing

# Feature Engineering

  * **Feature selection**
    * Removing uninformative features
  * **Feature extraction**
    * Creating new features from existing features
  * **Feature transformation**
    * Modifying existing features to better suit our objectives
    * Encoding of categorical features as dummies
 
When modeling, best practice is to perform a rigorous examination of your data before beginning feature engineering and feature selection. This process is important. Not only does it help you understand your data, what it's telling you, and what it's _not_ telling you, but it also can give you clues that help you create new features. 

### Drop unwanted features (Based on Domain Knowledge)

In [ ]:
df.head(1)

In [ ]:
df.columns

In [ ]:
df.drop(['zipcodestart', 'zipcodeend', 'neighborhoodstart', 'neighborhoodend'], axis=1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
#df.to_csv("cyclistic.csv", index=False)

### Reducing features after performing selection

In [ ]:
df.head()

In [ ]:
df2 = df[['lotarea']]

In [ ]:
df2

In [ ]:
df2.to_csv("ameshousingmod.csv", index=False)

**==============================================================================================================**

## Rename columns

In [ ]:
df.columns

In [ ]:
df = df.rename(columns =  {'species_Bream': 'bream',
                           'species_Rare': 'rare'
                          })

In [ ]:
# Rename columns as needed
 
df = df.rename(columns =  {'countryregionname': 'country',
                          'homeownerflag': 'homeowner',
                          'numbercarsowned': 'cars',
                          'numberchildrenathome':'child',
                          'avemonthspend': 'spend'
                          })

# Display all column names after the update
### YOUR CODE HERE ### 
df.columns

In [ ]:
#Method 3: Using a new list of column names

# Creating a list of new columns
df_cols = ['RankSeason', 'RankPlayoffs', 'OOBP', 'OSLG', 
           'MIRankSeason', 'MIRankPlayoffs', 
           'MIOOBP', 'MIOSLG', 'Team', 'League', 'Year', 
           'RS', 'RA', 'W', 'OBP', 'SLG', 'BA', 'Playoffs', 'G'
          ]

# printing the columns
# before renaming
print(df2.columns)

# Renaming the columns
df2.columns = df_cols

# printing the columns
# after renaming
print(df2.columns)


In [ ]:
# make all column headers in pandas data frame lower case

df.columns = map(str.lower, df.columns)

In [ ]:
df.columns

In [ ]:
# remove special character
df.columns = df.columns.str.replace(' ', '')

In [ ]:
# remove special character
df.columns = df.columns.str.replace('_', '')

In [ ]:
# remove special character
df.columns = df.columns.str.replace('-', '')

In [ ]:
# remove special character
df.columns = df.columns.str.replace('.', '')

In [ ]:
df.head()

In [ ]:
#df.to_csv("cyclistic.csv", index=False)

**==============================================================================================================**

## Rearrange columns

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
cols = ['id', 'limitbal', 'gender', 'educ', 'marital', 'age', 'medianpay', 'medianbillamt', 'medianpayamt', 'default'
        ]

In [ ]:
df = df[cols]

In [ ]:
df.head()

In [ ]:
#df.to_csv("creditcard.csv", index=False)

**==============================================================================================================**

## Data Types

<p>The last step in data cleaning is checking and making sure that all data is in the correct format (int, float, text or other).</p>

In Pandas, we use:

<p><b>.dtype()</b> to check the data type</p>
<p><b>.astype()</b> to change the data type</p>


In [ ]:
df.dtypes

In [ ]:
df[["extra", "mtatax","tollsamount","improvementsurcharge"]] = df[["extra", "mtatax","tollsamount","improvementsurcharge"]].astype('object')

In [ ]:
#df[["bore", "stroke"]] = df[["bore", "stroke"]].astype("float")

In [ ]:
df[["tpeppickupdatetimehour", "tpepdropoffdatetimehour"]] = df[["tpeppickupdatetimehour", "tpepdropoffdatetimehour"]].astype("int64")

In [ ]:
df.info()

In [ ]:
df.head(1)

**==============================================================================================================**

## Values Replacement

In [ ]:
df.head()

In [ ]:
df["privacylawseffective"].value_counts().to_frame()

In [ ]:
# Replace with same value for multiple
df2 = df.replace(to_replace=['Seattle', 'SouthCarolina',
       'SouthCentral', 'Southeast', 'Spokane', 'StLouis', 'Syracuse',
       'Tampa', 'TotalUS', 'WestTexNewMexico'], value='East')

In [ ]:
df["legendary"] = df["legendary"].replace(to_replace=False, value=0)

In [ ]:
df["legendary"] = df["legendary"].replace(to_replace=True, value=1)

In [ ]:
df["privacylawseffective"] = df["privacylawseffective"].replace(to_replace=np.nan, value=1.00)

In [ ]:
df2

In [ ]:
df2["region"].value_counts().to_frame()

In [ ]:
df["region"].replace(to_replace=['5e9e3032383ecb554034e7c9'],
           value=['c9'], inplace=True)

In [ ]:
df.head()

In [ ]:
#df.to_csv("anonymitypoll.csv", index=False)

**==============================================================================================================**

## Treat Missing Values

<b>How to deal with missing data?</b>

<ol>
    <li>Drop data<br>
        a. Drop the whole row<br>
        b. Drop the whole column
    </li>
    <li>Replace data<br>
        a. Replace it by mean<br>
        b. Replace it by frequency<br>
        c. Replace it based on other functions
    </li>
</ol>

For easier detection of missing values, pandas provides the `isna()`, `isnull()`, and `notna()` functions. For more information on pandas missing values please check out this documentation).

There are several options for dealing with missing values. We will use 'Lot Frontage' feature to analyze for missing values.

1. We can drop the missing values, using `dropna()` method.

2. We can drop the whole attribute (column), that contains missing values, using the `drop()` method.

3. We can replace the missing values (zero, the mean, the median, etc.), using `fillna()` method.

In [ ]:
df.isnull().sum()

In [ ]:
pd.Series(df.isnull().sum()).sort_values().to_frame()

In [ ]:
pd.Series(df.isnull().sum()).nlargest(n=9).to_frame()

In [ ]:
list(df.isnull().sum().nlargest(n=9).index)

In [ ]:
df.dropna(axis=0, inplace=True)

In [ ]:
df.reset_index(drop=True, inplace=True)

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
#df.to_csv("tiktokmod.csv", index=False)

### Data Imputation

Imputation is the act of replacing missing data with statistical estimates of the missing values. The goal of any imputation technique is to produce a **complete dataset** that can be used to train machine learning models.


In [ ]:
df.merchantsuburb.value_counts()

### Pandas Method

In [ ]:
df['tpepdropoffdatetimehour'] = df['tpepdropoffdatetimehour'].replace(np.nan,3.00)

In [ ]:
df["normalized-losses"].replace(np.nan, avg_norm_loss, inplace=True)

In [ ]:
imputation_dict = df[['Age']].mean().to_dict()

imputation_dict

In [ ]:
# Replace missing data

df.fillna(imputation_dict, inplace=True)

In [ ]:
df[['WorkWeekHrs']].hist();

**==============================================================================================================**

## Feature Engine Method

### Mean / Median Imputation

In [ ]:
# To perform median imputation, we specify the
# imputation strategy

imputer = MeanMedianImputer(imputation_method="median", variables=['ConvertedComp'])

In [ ]:
# let's do mean imputation over 2 of the 3 numerical variables

imputer = MeanMedianImputer(
    imputation_method="mean",
    variables=["Age", "WorkWeekHrs"],
)

In [ ]:
imputer.fit(df)

In [ ]:
imputer.variables_

In [ ]:
imputer.imputer_dict_

In [ ]:
df2 = imputer.transform(df)

In [ ]:
df2.isnull().sum()

In [ ]:
df2.head()

In [ ]:
#df2.to_csv("surveydata.csv", index=False)

### Arbitrary Imputation

In [ ]:
df["CodeRevHrs"].isnull().sum()/len(df)

In [ ]:
df.describe(include=["int", "float"])

In [ ]:
# we call the imputer from Feature-engine
# pecifying the arbitrary value

imputer = ArbitraryNumberImputer(arbitrary_number=0.00, variables=['internetuse', 'smartphone', 
                                                                   'worryaboutinfo', 'anonymitypossible',
                                                                   'triedmaskingidentity'])

In [ ]:
# we fit the imputer

imputer.fit(df)

In [ ]:
# we see that the imputer found the numerical variables

imputer.variables_

In [ ]:
# here we can see the arbitrary value

imputer.arbitrary_number

In [ ]:
# Feature-engine returns a dataframe

df2 = imputer.transform(df)

In [ ]:
df2.isnull().sum()

In [ ]:
df2.head()

In [ ]:
#df2.to_csv("anonymitypoll.csv", index=False)

### Frequent Category Imputation

In [ ]:
df.columns

In [ ]:
df.head(1)

In [ ]:
df["internetuse"].value_counts().to_frame()

In [ ]:
df["merchantstate"].value_counts().to_frame()

In [ ]:
# We specify how we want to impute
# the categorical variables.

imputer = CategoricalImputer(imputation_method="frequent", variables=["LandingPad"])

In [ ]:
# we fit the imputer

imputer.fit(df)

In [ ]:
imputer.variables_

In [ ]:
# here we can see the values that will be used
# to replace NA for each variable

imputer.imputer_dict_

In [ ]:
df2 = imputer.transform(df)

In [ ]:
df2.isnull().sum()

In [ ]:
df2.head()

In [ ]:
#df2.to_csv("spacexmod.csv", index=False)

### Missing Category Imputation

In [ ]:
df.columns

In [ ]:
# we call the imputer from feature engine.
# By default it performs imputation with a string missing.

imputer = CategoricalImputer(imputation_method='missing', variables=['merchantsuburb', 'merchantstate'])

In [ ]:
# we fit the imputer

imputer.fit(df)

In [ ]:
# we see that the imputer found the categorical variables

imputer.variables_

In [ ]:
# feature-engine returns a dataframe

df2 = imputer.transform(df)

In [ ]:
df2

In [ ]:
df2.isnull().sum()

In [ ]:
#df2.to_csv("bikesharemod.csv", index=False)

### Missing Indicator

In [ ]:
df.columns

In [ ]:
# we call the imputer from feature-engine
# the argument how allows us to determine if we want
# to add missing indicators to all variables, or only to
# those that show missing data in the train set

imputer = AddMissingIndicator(missing_only=True, variables=['start_station_name', 'start_station_id',
                                                           'end_station_name', 'end_station_id'])

In [ ]:
# we fit the imputer

imputer.fit(df)

In [ ]:
# the attribute `variables` shows the variables entered by the user, in this
# case None

imputer.variables

In [ ]:
# this attribute stores the variables, numerical and categorical,
# that had missing data in the train set

imputer.variables_

In [ ]:
# feature-engine returns a dataframe
# with the additional features

# no need to contatenate!!

df2 = imputer.transform(df)

In [ ]:
df2

In [ ]:
df2["start_station_id_na"].value_counts()

In [ ]:
#df2.to_csv("bikesharemod.csv", index=False)

### Missing value imputation: DropMissingData

In [ ]:
df["label"].value_counts(dropna=False)

In [ ]:
imputer = DropMissingData(missing_only = True, threshold = None,  
                          variables: = None)

In [ ]:
imputer.fit(X_train)

In [ ]:
# variables from which observations with NA will be deleted

imputer.variables_

### Random Sample Imputation

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
# we call the imputer from feature-engine

imputer = RandomSampleImputer(variables=['age', 'conservativeness', 'infooninternet', 'privacyimportance'], 
                              random_state=0, seed="general", seeding_method = 'add')

In [ ]:
# we fit the imputer

imputer.fit(df)

In [ ]:
# we see that the imputer selected all the variables, numerical
# and categorical

imputer.variables_

In [ ]:
df2 = imputer.transform(df)

In [ ]:
df2

In [ ]:
df2.isnull().sum()

In [ ]:
#df2.to_csv("anonymitypoll.csv", index=False)

**==============================================================================================================**

## Treat Duplicate Values

In [ ]:
df.duplicated(keep='first').sum()

In [ ]:
#identify duplicate rows
duplicateRows = df[df.duplicated(keep='last')]

In [ ]:
duplicateRows

In [ ]:
df.drop_duplicates(ignore_index=True, inplace=True)

In [ ]:
df

In [ ]:
#df.to_csv("Pokemon.csv", index=False)

**==============================================================================================================**

## Treat Outliers

In statistics, an outlier is an observation point that is distant from other observations. An outlier can be due to some mistakes in data collection or recording, or due to natural high variability of data points. How to treat an outlier highly depends on our data or the type of analysis to be performed. Outliers can markedly affect our models and can be a valuable source of information, providing us insights about specific behaviours.

There are many ways to discover outliers in our data. We can do Uni-variate analysis (using one variable analysis) or Multi-variate analysis (using two or more variables). One of the simplest ways to detect an outlier is to inspect the data visually, by making box plots or scatter plots. 

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
# Draw Box Plots

plt.figure(figsize=(20,7))
sns.boxplot(data=df.select_dtypes(include=np.number))
plt.show()

In [ ]:
# Draw Box Plots

plt.figure(figsize=(20,7))
sns.boxplot(data=df.select_dtypes(include="object"))
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x="height", data=df, orient="h")
#plt.xlim(0, 3600)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x="countfloorspreeq", orient="h", data=df)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df.age, orient="h")
plt.xlim(0, 100)
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df.area, orient="h")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df.videolikecount, orient="h")
plt.show()

In [ ]:
df.columns

### Removing outliers - outlier trimming

In [ ]:
outliertrimmer = OutlierTrimmer(capping_method="iqr", fold=1.5, tail="right", 
                                variables=['totalsteps' , 'calories'])

In [ ]:
outliertrimmer = OutlierTrimmer(capping_method="iqr", fold=1.5, tail="both", 
                                variables=['tripdistance', 'fareamount', 'tipamount', 'totalamount'])

In [ ]:
## Normal distribution
outliertrimmer = OutlierTrimmer(
    variables=["MedInc", "HouseAge", "Population"],
    capping_method="gaussian",
    tail="both",
    fold=3
)

In [ ]:
outliertrimmer = OutlierTrimmer(capping_method="quantiles", fold=0.05, tail="both", 
                                variables=['rainfall', 'windgustspeed', 'windspeed9am', 'windspeed3pm'])

In [ ]:
outliertrimmer.fit(df)

In [ ]:
outliertrimmer.left_tail_caps_

In [ ]:
outliertrimmer.right_tail_caps_

In [ ]:
df2 = outliertrimmer.transform(df)
df2

In [ ]:
df2.reset_index(drop=True, inplace=True)

In [ ]:
df2

In [ ]:
df.kmperdrivingday.hist();

In [ ]:
df2.kmperdrivingday.hist();

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df2.starting, orient="h")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df2.hours, orient="h")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df2.hours, orient="h")
plt.show()

In [ ]:
df2.describe()

In [ ]:
#df2.to_csv("wellnessmod.csv", index=False)

## Capping / Censoring outliers

The Winsorizer() caps maximum and / or minimum values of a variable.

The Winsorizer() works only with numerical variables

In [ ]:
df.columns

In [ ]:
capper = Winsorizer(
    variables=["worst smoothness", "worst texture"],
    capping_method="gaussian",
    tail="both",
    fold=2,
)

In [ ]:
capper = Winsorizer(
    variables=['veryactiveminutes', 'fairlyactiveminutes', 'lightlyactiveminutes', 'sedentaryminutes'],
    capping_method="iqr",
    tail="right",
    fold=1.5,
)


In [ ]:
capper = Winsorizer(
    variables=["worst smoothness", "worst texture"],
    capping_method="quantiles",
    tail="both",
    fold=0.05,
)

In [ ]:
capper = Winsorizer(
    variables=["worst smoothness", "worst texture"],
    capping_method="mad",
    tail="both",
    fold=0.05,
)

In [ ]:
capper.fit(df)

In [ ]:
capper.left_tail_caps_

In [ ]:
capper.right_tail_caps_

In [ ]:
df2 = capper.transform(df)

In [ ]:
df2.describe()

In [ ]:
#df2.to_csv("wellnessmod.csv", index=False)

## ArbitraryOutlierCapper

The ArbitraryOutlierCapper caps the minimum and maximum values by a value determined by the user.

In [ ]:
df.columns

In [ ]:
# let's find out the maximum values
df[['countfloorspreeq', 'age', 'area', 'height']].max()

In [ ]:
df[['countfloorspreeq', 'age', 'area', 'height']].min()

In [ ]:
capper = ArbitraryOutlierCapper(
    max_capping_dict={'countfloorspreeq': 4.0, 'age': 70.0, 'area': 200.0, 'height': 10.0},
    min_capping_dict=None,
)


In [ ]:
capper = ArbitraryOutlierCapper(
    max_capping_dict=None,
    min_capping_dict={'fareamount': 0.00, 'totalamount': 0.00},
)


In [ ]:
capper.fit(df)

In [ ]:
capper.left_tail_caps_

In [ ]:
capper.right_tail_caps_

In [ ]:
df2 = capper.transform(df)

In [ ]:
df2

In [ ]:
df2.describe()

In [ ]:
#df2.to_csv("earthquakemod.csv", index=False)

**==============================================================================================================**

## Rounding Values

In [ ]:
df.col

In [ ]:
###pandas.DataFrame.round
df[['internetuserate']] = df[['internetuserate']].round(decimals=0)

In [ ]:
#df.to_csv(".csv", index=False)

**==============================================================================================================**

## One-hot encoding

There are three unique values: France, Spain, and Germany. Let's encode this data so it can be represented using Boolean features. We'll use a pandas function called `pd.get_dummies()` to do this.

When we call `pd.get_dummies()` on this feature, it will replace the `Geography` column with three new Boolean columns--one for each possible category contained in the column being dummied. 

When we specify `drop_first=True` in the function call, it means that instead of replacing `Geography` with three new columns, it will instead replace it with two columns. We can do this because no information is lost from this, but the dataset is shorter and simpler.  

In this case, we end up with two new columns called `Geography_Germany` and `Geography_Spain`. We don't need a `Geography_France` column. Why not? Because if a customer's values in `Geography_Germany` and `Geography_Spain` are both 0, we'll know they're from France! 

In [ ]:
df.info()

In [ ]:
df.describe(include="object")

In [ ]:
list(df.describe(include="object"))

In [ ]:
df["verifiedstatus"].value_counts().to_frame()

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x=df.extra, y=df.duration, data=df, ci=None, estimator=mean)
plt.title("")
plt.show()

In [ ]:
df["major"].value_counts().to_frame()

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x=df.mtatax, y=df.duration, data=df, ci=None, estimator=mean)
plt.title("")
plt.show()

In [ ]:
df["tollsamount"].value_counts().to_frame()

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x=df.tollsamount, y=df.duration, data=df, ci=None, estimator=mean)
plt.title("")
plt.show()

In [ ]:
df["improvementsurcharge"].value_counts().to_frame()

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(x=df.improvementsurcharge, y=df.duration, data=df, ci=None, estimator=mean)
plt.title("")
plt.show()

In [ ]:
df["pulocationid"].value_counts().to_frame().head(10)

In [ ]:
df["dolocationid"].value_counts().to_frame().head(10)

In [ ]:
# Method 1: Separate the categorical
df.columns

In [ ]:
df_cat = df[['Bldg Type', 'House Style']]

In [ ]:
df_cat

In [ ]:
df_cat = pd.get_dummies(data=df_cat, drop_first=True, prefix=None)

In [ ]:
df_cat

In [ ]:
df2 = pd.concat([df,df_cat], axis=1)

In [ ]:
df2

In [ ]:
df2.drop(['Bldg Type', 'House Style'], axis=1, inplace=True)

In [ ]:
#df.to_csv(".csv", index=False)

In [ ]:
# Method 2: Use pandas dummies
df.columns

In [ ]:
list(df.select_dtypes(['object']).columns)

In [ ]:
list(df.select_dtypes(['bool']).columns)

In [ ]:
df2 = pd.get_dummies(df, prefix=['vendorid'], 
                     columns = ['vendorid'], drop_first=True)

In [ ]:
df2.head()

In [ ]:
#df2.to_csv("nyctaximod.csv", index=False)

In [ ]:
# Method 3: One-hot encode the categorical variables as needed and save resulting dataframe in a new variable
df2 = pd.get_dummies(df, prefix=['cycle', 'gear'], columns = ['cyl', 'gear'], drop_first=False)

# Display the new dataframe
df2.head()

In [ ]:
df2 = pd.concat([df,df_cat], axis=1)

In [ ]:
df2

## One Hot Encoding - Feature-engine

Just like imputation, all methods of categorical encoding should be performed over the training set, and then propagated to the test set. 

Why? 

Because these methods will "learn" patterns from the train data, and therefore you want to avoid leaking information and overfitting. But more importantly, because we don't know whether in future / live data, we will have all the categories present in the train data, or if there will be more or less categories. Therefore, we want to anticipate this uncertainty by setting the right processes right from the start. We want to create transformers that learn the categories from the train set, and used those learned categories to create the dummy variables in both train and test sets.

In [ ]:
df.columns

In [ ]:
df.describe(include="object")

In [ ]:
list(df.describe(include="object"))

In [ ]:
# set up encoder

encoder = OneHotEncoder(
    variables=['legendary'],  # alternatively pass a list of variables
    drop_last=False,  # to return k-1, use drop=false to return k dummies
)

In [ ]:
# fit the encoder (finds categories)

encoder.fit(df)

In [ ]:
# automatically found numerical variables

encoder.variables_

In [ ]:
# we observe the learned categories

encoder.encoder_dict_

In [ ]:
# transform the data sets

df2 = encoder.transform(df)

In [ ]:
df2.head()

In [ ]:
# we can retrieve the feature names as follows:

encoder.get_feature_names_out()

In [ ]:
df2.head()

In [ ]:
#df2.to_csv("fish.csv", index=False)

## Ordinal Encoding - Feature-engine

Ordinal encoding consist in replacing the categories by integers from 1 to n (or 0 to n-1, depending the implementation), where n is the number of distinct categories of the variable.

The numbers are assigned arbitrarily. This encoding method allows for quick benchmarking of machine learning models. It is also suitable for tree based machine learning algorithms.


In [ ]:
# let's explore the unique categories
df["color"].unique()

In [ ]:
ordinal_enc = OrdinalEncoder(
    encoding_method="arbitrary",
    variables=["color"],
)


In [ ]:
ordinal_enc.fit(df)

In [ ]:
# in the encoder dict we can observe the numbers
# assigned to each category for all the indicated variables

ordinal_enc.encoder_dict_

In [ ]:
# this is the list of variables that the encoder will transform

ordinal_enc.variables_

In [ ]:
df2 = ordinal_enc.transform(df)

In [ ]:
df2.head()

In [ ]:
#df2.to_csv("earthquakemod.csv", index=False)

## Count or frequency encoding - Feature-engine

In count encoding we replace the categories by the count of the observations that show that category in the dataset. Similarly, we can replace the category by the frequency -or percentage- of observations in the dataset. That is, if 10 of our 100 observations show the colour blue, we would replace blue by 10 if doing count encoding, or by 0.1 if replacing by the frequency. These techniques capture the representation of each label in a dataset, but the encoding may not necessarily be predictive of the outcome. These are however, very popular encoding methods in Kaggle competitions.

The assumption of this technique is that the number observations shown by each variable is somewhat informative of the predictive power of the category.


In [ ]:
# let's explore the unique categories
df["color"].unique()

In [ ]:
count_enc = CountFrequencyEncoder(
    encoding_method="count",  # to do frequency ==> encoding_method='frequency'
    variables=["color"]
)

In [ ]:
count_enc.fit(df)

In [ ]:
# in the encoder dict we can observe the number of
# observations per category for each variable

count_enc.encoder_dict_

In [ ]:
df2 = count_enc.transform(df)

In [ ]:
df2.head()

In [ ]:
#df2.to_csv("earthquakemod.csv", index=False)

## Ordered Integer Encoding

Ordering the categories according to the target means assigning a number to the category from 1 to k, where k is the number of distinct categories in the variable, but this numbering is informed by the mean of the target for each category.

**Note**

If the argument `variables` is left to None, then the encoder will automatically identify all categorical variables. Is that not sweet?

The encoder will not encode numerical variables. So if some of your numerical variables are in fact categories, you will need to re-cast them as object before using the encoder.

Finally, if there is a label in the test set that was not present in the train set, the encoder will through and error, to alert you of this behaviour.

In [ ]:
# let's separate into training and testing set

X_train, X_test, y_train, y_test = train_test_split(
    df[["color"]],  # predictors
    df["price"],  # target
    test_size=0.2,  # percentage of obs in test set
    random_state=0,
)  # seed to ensure reproducibility

In [ ]:
ordinal_enc = OrdinalEncoder(
    # NOTE that we indicate ordered in the encoding_method, otherwise it assings numbers arbitrarily
    encoding_method="ordered",
    variables=["color"],
)

In [ ]:
# when fitting the transformer, we need to pass the target as well
# just like with any Scikit-learn predictor class

ordinal_enc.fit(X_train, y_train)

In [ ]:
# in the encoder dict we can observe each of the top categories
# selected for each of the variables

ordinal_enc.encoder_dict_

In [ ]:
# this is the list of variables that the encoder will transform

ordinal_enc.variables_

In [ ]:
X_train = ordinal_enc.transform(X_train)
X_test = ordinal_enc.transform(X_test)

In [ ]:
# let's explore the result
X_train.head()

## Mean Encoding or Target Encoding

Mean encoding implies replacing the category by the average target value for that category. For example, if we have the variable city, with categories London, Manchester and Bristol, and we want to predict the default rate, if the default rate for London is 30% we replace London by 0.3, if the default rate for Manchester is 20% we replace Manchester by 0.2 and so on.

If using Feature-Engine, instead of pandas, we do not need to keep the target variable in the training dataset.

In [ ]:
df.dtypes[df.dtypes == 'object']

In [ ]:
# let's separate into training and testing set

X_train, X_test, y_train, y_test = train_test_split(
    df[["cut", "color", "clarity"]],
    df["price"],  # target
    test_size=0.2,  # percentage of obs in test set
    random_state=0,
)  # seed to ensure reproducibility


In [ ]:
X_train.shape, X_test.shape

In [ ]:
mean_enc = MeanEncoder(variables=["cut", "color", "clarity"], smoothing="auto")

In [ ]:
# when fitting the transformer, we need to pass the target as well
# just like with any Scikit-learn predictor class

mean_enc.fit(X_train, y_train)

In [ ]:
# in the encoder dict we see the target mean assigned to each
# category for each of the selected variables

mean_enc.encoder_dict_

In [ ]:
# this is the list of variables that the encoder will transform

mean_enc.variables_

In [ ]:
X_train = mean_enc.transform(X_train)
X_test = mean_enc.transform(X_test)

In [ ]:
# let's explore the result
X_train.head()

## Weight  of evidence

Weight of Evidence (WoE) was developed primarily for the credit and financial industries to help build more predictive models to evaluate the risk of loan default. That is, to predict how likely the money lent to a person or institution is to be lost. Thus, Weight of Evidence is a measure of the "strength” of a grouping technique to separate good and bad risk (default). 

- WoE will be 0 if the P(Goods) / P(Bads) = 1, that is, if the outcome is random for that group.
- If P(Bads) > P(Goods) the odds ratio will be < 1 and,
- WoE will be < 0 if,  P(Goods) > P(Bads).

WoE is well suited for Logistic Regression, because the Logit transformation is simply the log of the odds, i.e., ln(P(Goods)/P(Bads)). Therefore, by using WoE-coded predictors in logistic regression, the predictors are all prepared and coded to the same scale, and the parameters in the linear logistic regression equation can be directly compared.

The WoE transformation has three advantages:

- It creates a monotonic relationship between the target and the independent variables.
- It orders the categories on a "logistic" scale which is natural for logistic regression
- The transformed variables can then be compared because they are on the same scale. Therefore, it is possible to determine which one is more predictive.

The WoE also has a limitation:

- Prone to cause over-fitting

**==============================================================================================================**

## One Hot Encoding of Frequent Categories

We learned in Section 4 that high cardinality and rare labels may result in certain categories appearing only in the train set, therefore causing over-fitting, or only in the test set, and then our models wouldn't know how to score those observations.

We also learned in the previous lecture on one hot encoding, that if categorical variables contain multiple labels, then by re-encoding them with dummy variables we will expand the feature space dramatically.

**In order to avoid these complications, we can create dummy variables only for the most frequent categories**

This procedure is also called one hot encoding of top categories.

### Advantages of OHE of top categories

- Straightforward to implement
- Does not require hrs of variable exploration
- Does not expand massively the feature space
- Suitable for linear models


### Limitations

- Does not add any information that may make the variable more predictive
- Does not keep the information of the ignored labels


Often, categorical variables show a few dominating categories while the remaining labels add little information. Therefore, OHE of top categories is a simple and useful technique.

### Note

The number of top variables is set arbitrarily. In the KDD competition the authors selected 10, but it could have been 15 or 5 as well. This number can be chosen arbitrarily or derived from data exploration.


## OHE with pandas and NumPy

In [ ]:
df[["cut", "color", "clarity"]]

In [ ]:
# let's explore the unique categories
df["cut"].unique()

In [ ]:
# let's find the top 10 most frequent categories for the variable 'Neighborhood'

df["cut"].value_counts().sort_values(ascending=False).head(10)

In [ ]:
# let's explore the unique categories
df["color"].unique()

In [ ]:
# let's find the top 10 most frequent categories for the variable 'Neighborhood'

df["color"].value_counts().sort_values(ascending=False).head(10)

In [ ]:
# let's explore the unique categories
df["clarity"].unique()

In [ ]:
# let's find the top 10 most frequent categories for the variable 'Neighborhood'

df["clarity"].value_counts().sort_values(ascending=False).head(10)

In [ ]:
df2 = pd.get_dummies(df, prefix=['cut'], 
                     columns = ['cut'], drop_first=True)

In [ ]:
#df2.to_csv("earthquakemod.csv", index=False)

## One hot encoding of top categories with Feature-Engine

In [ ]:
df.info()

In [ ]:
df.describe(include="object")

In [ ]:
list(df.describe(include="object"))

In [ ]:
df["type2"].value_counts().to_frame()

In [ ]:
df["color"].value_counts().to_frame()

In [ ]:
df["clarity"].value_counts().to_frame()

In [ ]:
df["vendorid"].value_counts().to_frame()

In [ ]:
df["pulocationid"].value_counts().to_frame().head(5)

In [ ]:
df["dolocationid"].value_counts().to_frame().head(5)

In [ ]:
df["paymenttype"].value_counts().to_frame()

In [ ]:
ohe_enc = OneHotEncoder(
    top_categories=3,  # you can change this value to select more or less variables
    # we can select which variables to encode
    variables=["type2"],
    drop_last=False,
)


In [ ]:
ohe_enc.fit(df)

In [ ]:
# in the encoder dict we can observe each of the top categories
# selected for each of the variables

ohe_enc.encoder_dict_

In [ ]:
# this is the list of variables that the encoder will transform

ohe_enc.variables_

In [ ]:
df2 = ohe_enc.transform(df)

In [ ]:
df2

In [ ]:
df["type2"].value_counts().to_frame()

In [ ]:
df2["type2"].value_counts()

In [ ]:
df2["paymenttype_2"].value_counts()

In [ ]:
#df2.to_csv("Pokemon.csv", index=False)

**==============================================================================================================**

## Engineering Rare Categories

Rare values are categories within a categorical variable that are present only in a small percentage of the observations. There is no rule of thumb to determine how small is a small percentage, but typically, any value below 5 % can be considered rare.

As we discussed in section 3 of the course, Infrequent labels are so few, that it is hard to derive reliable information from them. But more importantly, if you remember from section 3, infrequent labels tend to appear only on train set or only on the test set:

- If only on the train set, they may cause over-fitting
- If only on the test set, our machine learning model will not know how to score them

Therefore, to avoid this behaviour, we tend to group those into a new category called 'Rare' or 'Other'.

Rare labels can appear in low or highly cardinal variables. There is no rule of thumb to determine how many different labels are considered high cardinality. It depend as well on how many observations there are in the dataset. In a dataset with 1,000 observations, 100 labels may seem a lot, whereas in a dataset with 100,000 observations it may not be so high.

Highly cardinal variables tend to have many infrequent or rare categories, whereas low cardinal variables, may have only 1 or 2 rare labels.

### Note the following:

**Note that grouping infrequent labels or categories under a new category called 'Rare' or 'Other' is the common practice in machine learning for business.**

- Grouping categories into rare for variables that show low cardinality may or may not improve model performance, however, we tend to re-group them into a new category to smooth model deployment.

- Grouping categories into rare for variables with high cardinality, tends to improve model performance as well.

In [ ]:
df.columns

In [ ]:
list(df.select_dtypes("object"))

In [ ]:
def find_non_rare_labels(df, variable, tolerance):

    temp = df.groupby([variable])[variable].count() / len(df)

    non_rare = [x for x in temp.loc[temp > tolerance].index.values]

    return non_rare

In [ ]:
# non rare labels (min 5%)
find_non_rare_labels(df, "startstationname", 0.008)

In [ ]:
df["startstationname"].value_counts().to_frame()

In [ ]:
# non rare labels (min 5%)
find_non_rare_labels(df, "startstationid", 0.008)

In [ ]:
df["startstationid"].value_counts().to_frame()

In [ ]:
# non rare labels (min 5%)
find_non_rare_labels(df, "endstationname", 0.009)

In [ ]:
df["endstationname"].value_counts().to_frame()

In [ ]:
# non rare labels (min 5%)
find_non_rare_labels(df, "endstationid", 0.009)

In [ ]:
df["endstationid"].value_counts().to_frame()

## Encoding Rare Labels with Feature-Engine

In [ ]:
# Rare value encoder
rare_encoder = RareLabelEncoder(
    tol=0.008,  # minimal percentage to be considered non-rare
    n_categories=2,  # minimal number of categories the variable should have to re-group rare categories
    variables=['startstationname','startstationid'],  # variables to re-group
    missing_values="ignore"
)

In [ ]:
# Rare value encoder
rare_encoder = RareLabelEncoder(
    tol=0.009,  # minimal percentage to be considered non-rare
    n_categories=2,  # minimal number of categories the variable should have to re-cgroup rare categories
    variables=[ "endstationname", "endstationid" ],  # variables to re-group
    missing_values="ignore"
)

In [ ]:
rare_encoder.fit(df)

In [ ]:
rare_encoder.variables_

In [ ]:
# the encoder_dict_ is a dictionary of variable: frequent labels pair
rare_encoder.encoder_dict_

In [ ]:
df2 = rare_encoder.transform(df)

In [ ]:
df2.head()

In [ ]:
df2[['endstationname']].apply(pd.Series.value_counts)

In [ ]:
df2.isnull().sum()

In [ ]:
#df2.to_csv("bikeshare.csv", index=False)

**==============================================================================================================**

# Discretization

Discretization is the process of transforming continuous variables into discrete variables by creating a set of contiguous intervals that span the range of the variable's values. Discretization is also called **binning**, where bin is an alternative name for interval.


### Discretization helps handle outliers and may improve the value spread in skewed variables

Discretization helps handle outliers by placing these values into the lower or higher intervals, together with the remaining inlier values of the distribution. Thus, these outlier observations no longer differ from the rest of the values at the tails of the distribution, as they are now all together in the same interval or bucket. In addition, by creating appropriate bins or intervals, discretization can help spread the values of a skewed variable across a set of bins with an equal number of observations.


### Discretization approaches

There are several approaches to transform continuous variables into discrete ones. Discretization methods fall into 2 categories: **supervised and unsupervised**. Unsupervised methods do not use any information other than the variable distribution to create the contiguous bins in which the values will be placed. Supervised methods typically use target information in order to create the bins or intervals.


####  Unsupervised discretization methods

- Equal width discretisation
- Equal frequency discretization
- K-means discretization

#### Supervised discretization methods

- Discretization using decision trees


## Equal width discretization

Equal width discretization divides the scope of possible values into N bins of the same width. The width is determined by the range of values in the variable and the number of bins we wish to use to divide the variable:

width = (max value - min value) / N

where N is the number of bins or intervals.

For example, if the values of the variable vary between 0 and 100, we create 5 bins like this: width = (100-0) / 5 = 20. The bins thus are 0-20, 20-40, 40-60, 80-100. The first and final bins (0-20 and 80-100) can be expanded to accommodate outliers (that is, values under 0 or greater than 100 would be placed in those bins as well).

There is no rule of thumb to define N; that is something to determine experimentally.

## Equal width discretisation with Feature-Engine

In [ ]:
df.columns

In [ ]:
# with feature-engine we can automate the process for many variables
# in one line of code

disc = EqualWidthDiscretiser(bins=3, variables = ['tpeppickupdatetimehour', 'tpepdropoffdatetimehour'])

disc.fit(df)

In [ ]:
# in the binner dict, we can see the limits of the intervals. 

disc.binner_dict_

In [ ]:
# transform train and test

df2 = disc.transform(df)

In [ ]:
df2[['tpeppickupdatetimehour','tpepdropoffdatetimehour']].head()

In [ ]:
df2[['tpeppickupdatetimehour']].value_counts().to_frame()

In [ ]:
df2[['tpepdropoffdatetimehour']].value_counts().to_frame()

## Equal frequency discretization

Equal frequency discretization divides the scope of possible values of the variable into N bins, where each bin carries the same amount of observations. This is particularly useful for skewed variables, as it spreads the observations over the different bins equally. We find the interval boundaries by determining the quantiles.

Equal frequency discretization using quantiles consists of dividing the continuous variable into N quantiles, where N to be defined by the user.

Equal frequency binning is straightforward to implement, and by spreading the values of the observations more evenly, it may help boost the algorithm's performance. This arbitrary binning may also disrupt the relationship with the target.

In [ ]:
df.columns

In [ ]:
df['tpeppickupdatetimehour'].hist();

In [ ]:
df['tpepdropoffdatetimehour'].hist();

In [ ]:
# with feature engine we can automate the process for many variables
# in one line of code

disc = EqualFrequencyDiscretiser(
    q=3,
    variables = ['tpeppickupdatetimehour', 'tpepdropoffdatetimehour'],
    return_boundaries=False,
)


In [ ]:
disc.fit(df)

In [ ]:
disc.binner_dict_

In [ ]:
df2 = disc.transform(df)
df2.head()

In [ ]:
df2["tpeppickupdatetimehour"].value_counts().sort_index().plot.bar()
plt.show()

In [ ]:
df2["tpepdropoffdatetimehour"].value_counts().sort_index().plot.bar()
plt.show()

## Arbitrary discretization

Frequently, when engineering variables in a business setting, the business experts determine the intervals in which they think the variable should be divided so that it makes sense for the business. Typical examples include the discretization of variables like age and income.

In [ ]:
df.columns

In [ ]:
disc = ArbitraryDiscretiser(
    binning_dict = {
        "tpeppickupdatetimehour": [0, 8, 16, 24],
        "tpepdropoffdatetimehour": [0, 8, 16, 24]},
        return_object=False,
        return_boundaries=False
)

In [ ]:
df2 = disc.fit_transform(df)

In [ ]:
df2.head()

In [ ]:
df2["tpeppickupdatetimehour"].value_counts().sort_index().plot.bar()
plt.show()

In [ ]:
df2.duration.groupby(df2['tpeppickupdatetimehour']).mean().plot()
plt.show()

In [ ]:
df2["tpepdropoffdatetimehour"].value_counts().sort_index().plot.bar()
plt.show()

In [ ]:
df2.duration.groupby(df2['tpepdropoffdatetimehour']).mean().plot()
plt.show()

## Discretization plus Encoding

What shall we do with the variable after discretisation? should we use the buckets as a numerical variable? or should we use the intervals as categorical variable?

The answer is, you can do either.

If you are building decision tree based algorithms and the output of the discretisation are integers (each integer referring to a bin), then you can use those directly, as decision trees will pick up non-linear relationships between the discretised variable and the target.

If you are building linear models instead, the bins may not necessarily hold a linear relationship with the target. In this case, it may help improve model performance to treat the bins as categories and to one hot encoding, or target guided encodings like mean encoding, weight of evidence, or target guided ordinal encoding.

We can easily do so by combining feature-engine's discretisers and encoders.

In [ ]:
df.columns

In [ ]:
# set up the chosen discretiser
# to encode variables we need them returned as objects for feature-engine

disc = ArbitraryDiscretiser(
    binning_dict = {
        "tpeppickupdatetimehour": [0, 8, 16, 24],
        "tpepdropoffdatetimehour": [0, 8, 16, 24]},
        return_object=True,
        return_boundaries=False
)

In [ ]:
# discretize
df2 = disc.fit_transform(df)

In [ ]:
df2.head()

In [ ]:
#df2.to_csv("nyctaximod.csv", index=False)

## Encoding

In [ ]:
encoder = OrdinalEncoder(encoding_method="ordered",
                        variables=['tpeppickupdatetimehour', 'tpepdropoffdatetimehour'])

In [ ]:
X_train = encoder.fit_transform(X_train, y_train)

X_test = encoder.transform(X_test)

In [ ]:
# in the map, we map bin to position

encoder.encoder_dict_

**==============================================================================================================**

## Label Encoding

Label Encoding is a popular encoding technique for handling categorical variables. In this technique, each label is assigned a unique integer based on alphabetical ordering.

label_encoder object knows how to understand word labels. 

`label_encoder = LabelEncoder()`

Encode labels in column 'Country'. 

`df['Timely'] = label_encoder.fit_transform(df['Timely'])` 


In [ ]:
df.columns

In [ ]:
df.usertype.value_counts()

In [ ]:
df['raintomorrow'] = df['raintomorrow'].astype("category")

In [ ]:
# label_encoder object knows how to understand word labels. 
le = LabelEncoder()

# Encode labels in column 'Country'. 

df['usertype'] = le.fit_transform(df['usertype'])

In [ ]:
df["usertype"].value_counts()

In [ ]:
df.head()

In [ ]:
# Or do the mapping method

In [ ]:
df['usertype'] = df['usertype'].map({ 'Subscriber': 0, 'Customer': 1})

In [ ]:
df.usertype.value_counts()

In [ ]:
df.head(1)

In [ ]:
#df.to_csv("cyclistic.csv", index=False)

**==============================================================================================================**

# Feature Engineering

## Combine features with functions

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
# make a list with the features we want to combine

features = ['payamt1', 'payamt2', 'payamt3', 'payamt4', 'payamt5', 'payamt6'
     
]

In [ ]:
df[features].head()

In [ ]:
# list with functions to apply for combinations

math_func = ["sum", "prod", "mean", "median", "std", "max", "min", "count"]

In [ ]:
math_func = ["median"]

In [ ]:
# name of new features

new_feature_names = ["sum_f", "prod_f", "mean_f", "median_f", "std_f", "max_f", "min_f", "count_f"]

In [ ]:
new_feature_names = ["medianpay"]

In [ ]:
# automate feature combination with Feature-engine

create = MathFeatures(
    variables=features,
    func=math_func,
    new_variables_names=new_feature_names,
)

In [ ]:
create

In [ ]:
create = MathFeatures(variables=features, func="median", new_variables_names=["medianpayamt"], drop_original=True)

In [ ]:
# combine features

df2 = create.fit_transform(df)

In [ ]:
df2

In [ ]:
#df2.to_csv("creditcard.csv", index=False)

## Compare features to reference variable

In [ ]:
# features of interest
features = ["mean smoothness", "mean compactness", "mean concavity", "mean symmetry"]

# reference features
reference = ["mean radius", "mean area"]

In [ ]:
# combine multiple variables with multiple references

creator = RelativeFeatures(
    variables=features,
    reference=reference,
    func=["sub", "div"],
)


In [ ]:
df_t = creator.fit_transform(df)

**==============================================================================================================**

## Polynomial Features Transformation

In [ ]:
pr = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
pr

In [ ]:
pr = PolynomialFeatures(degree=5)
x_train_pr = pr.fit_transform(x_train[['horsepower']])
x_test_pr = pr.fit_transform(x_test[['horsepower']])
pr

In [ ]:
X_poly = pr.fit_transform(X)

In [ ]:
X_poly

In [ ]:
X_poly.shape

**==============================================================================================================**

# Feature extraction

Depending on your data, you may be able to create brand new features from your existing features. Oftentimes, features that you create yourself are some of the most important features selected by your model. Usually this is the case when you have both domain knowledge for the problem you're solving and the right combinations of data. 


In [ ]:
df.columns

In [ ]:
df["kmperdrivingday"] = df["drivenkmdrives"] / df["drivingdays"]

In [ ]:
df["kmperdrivingday"].describe()

In [ ]:
df["kmperdrivingday"] = df["kmperdrivingday"].replace(to_replace=np.inf, value=0)

In [ ]:
df["drives"].unique()

In [ ]:
df["drivingdays"].unique()

**==============================================================================================================**

## Create a list of conditions

In [ ]:
conditions = [
    (df['drives'] == 0),
    (df['drives'] > 0),
    (df['drives'] < 0)
    ]

# create a list of the values we want to assign for each condition
values = ['Shipped on Time', 'Shipped Early', 'Shipped Late']

# create a new column and use np.select to assign values to it using our lists as arguments
df['Ship Status'] = np.select(conditions, values)

In [ ]:
conditions = [
    (df['drives'] >= 100) & (df["drivingdays"] >=20),
    (df['drives'] < 100) & (df["drivingdays"] <20)
    ]

# create a list of the values we want to assign for each condition
values = [1,0]

# create a new column and use np.select to assign values to it using our lists as arguments
df['professionaldriver'] = np.select(conditions, values)

In [ ]:
df["professionaldriver"].value_counts(dropna=False)

In [ ]:
#df.to_csv("wazemod.csv", index=False)

**=============================================================================================================**

# Date Features from the datetime variable

In this notebook, we will see how we can easily derive many date-related features using the `dt` module from pandas.


## Features from the date part:

Below are some of the features that we can extract from the date part of the datetime variable off-the-shelf using [pandas](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#time-date-components):


- pandas.Series.dt.year
- pandas.Series.dt.quarter
- pandas.Series.dt.month
- pandas.Series.dt.isocalendar().week
- pandas.Series.dt.day
- pandas.Series.dt.day_of_week
- pandas.Series.dt.weekday
- pandas.Series.dt.dayofyear
- pandas.Series.dt.day_of_year

- pandas.Series.dt.is_month_start
- pandas.Series.dt.is_month_end
- pandas.Series.dt.is_quarter_start
- pandas.Series.dt.is_quarter_end
- pandas.Series.dt.is_year_start
- pandas.Series.dt.is_year_end
- pandas.Series.dt.is_leap_year
- pandas.Series.dt.days_in_month

We can use the features obtained with pandas to create even more features, such as:

- Semester
- Is Weekend?


In [ ]:
df.head()

In [ ]:
# Let's determine the type of data in the datetime variable.
df["timestamp"].dtypes

In [ ]:
df["tpeppickupdatetime"].dtypes

In [ ]:
# df['converted_col'] = pd.to_datetime(df.col, format='%Y-%m-%d %H:%M:%S')

In [ ]:
# This is how we parse date strings into datetime format:

df["timestamp"] = pd.to_datetime(df["timestamp"], errors='coerce')

df["timestamp"].head()

In [ ]:
# Unix Time Conversion:

df["timestamp"] = pd.to_datetime(df["timestamp"], unit='s')

df["timestamp"].head()

In [ ]:
df

In [ ]:
# This is how we parse date strings into datetime format:

df["tpeppickupdatetime"] = pd.to_datetime(df["tpeppickupdatetime"], errors='coerce')

df["tpeppickupdatetime"].head()

In [ ]:
# Create a current Date

df["currdate"] = pd.Timestamp.today()

In [ ]:
df.head()

In [ ]:
# This is how we parse date strings into datetime format:

df["tpepdropoffdatetime"] = pd.to_datetime(df["tpepdropoffdatetime"], format='%Y-%m-%d %H:%m:%S')

df["tpepdropoffdatetime"].head()

In [ ]:
# Extract the date part.

df["date_part"] = df["tpepdropoffdatetime"].dt.date

df["date_part"].head()

In [ ]:
# Extract the time part.
# (we don't need it for this demo,
# so I will not add it to the dataframe).

df["time_part"] = df["tpepdropoffdatetime"].dt.time

df["time_part"].head()

In [ ]:
# Extract year.

df["year"] = df["tpepdropoffdatetime"].dt.year

df["year"].head()

In [ ]:
# Extract year start and year end.

df["year_start"] = df["tpep_dropoff_datetime"].dt.is_year_start
df["year_end"] = df["tpep_dropoff_datetime"].dt.is_year_end

df[["year_start", "year_end", "tpep_dropoff_datetime"]].head()

In [ ]:
# Extract leap year.

df["year_leap"] = df["tpep_dropoff_datetime"].dt.is_leap_year

df["year_leap"].head()

In [ ]:
# Extract quarter from date variable - takes values 1 to 4.

df["quarter"] = df["tpep_dropoff_datetime"].dt.quarter

df[["date_part", "quarter"]].head()

In [ ]:
# Extract quarter start and end.

df["quarter_start"] = df["tpep_dropoff_datetime"].dt.is_quarter_start
df["quarter_end"] = df["tpep_dropoff_datetime"].dt.is_quarter_end

df[["quarter_start", "quarter_end", "date_part"]].head()

In [ ]:
# Extract month - 1 to 12.

df["month"] = df["tpepdropoffdatetime"].dt.month

df[["date_part", "month"]].head()

In [ ]:
# Number of days in a month.

df["days_in_month"] = df["tpep_dropoff_datetime"].dt.days_in_month

df[["days_in_month", "month"]].head()

In [ ]:
# Extract month start and end.

df["month_start"] = df["tpep_dropoff_datetime"].dt.is_month_start
df["month_end"] = df["tpep_dropoff_datetime"].dt.is_month_end

In [ ]:
df["month_start"].head()

In [ ]:
# Extract week of the year - varies from 1 to 52.

df["week"] = df["tpep_dropoff_datetime"].dt.isocalendar().week

df[["date_part", "week"]].head()

In [ ]:
# Day of the month - numeric from 1-31.

df["day"] = df["tpepdropoffdatetime"].dt.day

df[["date_part", "day"]].head()

In [ ]:
# Day of the week - from 0 to 6.

# It is assumed the week starts on Monday,
# denoted by 0, and ends on Sunday, denoted by 6.

df["dayofweek"] = df["tpepdropoffdatetime"].dt.dayofweek

df[["date_part", "dayofweek"]].head()

In [ ]:
# Day of the week - string (not useful for predictions,
# but since we are here...).

df["day_name"] = df["tpepdropoffdatetime"].dt.day_name()

df[["date_part", "day_name"]].head()

In [ ]:
# Day of the year - 1 to 365.

# I can't imagine when this feature would be
# useful. Maybe, if we had data for several years,
# to identify some repetitive pattern.

df["day_year"] = df["tpepdropoffdatetime"].dt.dayofyear

df[["date_part", "day_year"]].head()

In [ ]:
#df.to_csv("movielens.csv", index=False)

**==============================================================================================================**

# Time features from the datetime variable

In this notebook, we will see how we can easily derive many time-related features using the `dt` module from pandas.


## Features from the time part:

Below are some of the features that we can extract off-the-shelf using [pandas](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#time-date-components):

- pandas.Series.dt.hour
- pandas.Series.dt.minute
- pandas.Series.dt.second
- pandas.Series.dt.microsecond
- pandas.Series.dt.nanosecond

In [ ]:
# Let's determine the type of data in the datetime variable.

df["BirthDate"].dtypes

In [ ]:
# This is how we parse date strings into datetime format.

data["date"] = pd.to_datetime(df["tpep_dropoff_datetime"])

data["date"].head()

In [ ]:
# Extract time part.

# (We would normally not use this as a predictive feature,
# but it might be handy for data analysis).

df["time_part"] = df["tpepdropoffdatetime"].dt.time

df["time_part"].head()

In [ ]:
df["hour"] = df["tpepdropoffdatetime"].dt.hour
df["min"] = df["tpepdropoffdatetime"].dt.minute
df["sec"] = df["tpepdropoffdatetime"].dt.second

# We do not have micro and nano seconds in this dataset,
# but if we did, we can extract them as follows:

df["microsec"] = df["tpepdropoffdatetime"].dt.microsecond
df["nanosec"] = df["tpepdropoffdatetime"].dt.nanosecond

df.head()

In [ ]:
df.tail()

### Calculate difference between two dates in Days, Weeks, Months and Years

In [ ]:
df

In [ ]:
df["daydiff"] = (df['tpepdropoffdatetime'] - df['tpeppickupdatetime'])/np.timedelta64(1, 'D')

In [ ]:
df["monthdiff"] = (df['tpepdropoffdatetime'] - df['tpeppickupdatetime'])/np.timedelta64(1,'M')

In [ ]:
df["weekdiff"] = (df['tpepdropoffdatetime'] - df['tpeppickupdatetime'])/np.timedelta64(1,'W')

In [ ]:
df["yeardiff"] = (df['tpepdropoffdatetime'] - df['tpeppickupdatetime'])/np.timedelta64(1,'Y')

### Calculate difference between two dates in Hours, Minutes and Seconds

In [ ]:
df["hourdiff"] = (df['tpepdropoffdatetime'] - df['tpeppickupdatetime'])/np.timedelta64(1, 'h')

In [ ]:
df["mindiff"] = (df['tpepdropoffdatetime'] - df['tpeppickupdatetime'])/np.timedelta64(1, 'm')

In [ ]:
df["secdiff"] = (df['tpepdropoffdatetime'] - df['tpeppickupdatetime'])/np.timedelta64(1, 's')

In [ ]:
df["duration"] = (df['tpepdropoffdatetime'] - df['tpeppickupdatetime'])/np.timedelta64(1, 's')

In [ ]:
df

# Date and Time features with Feature-engine

We can automate the extraction of date and time features with Feature-engine, using the [DatetimeFeatures](https://feature-engine.readthedocs.io/en/latest/api_doc/datetime/DatetimeFeatures.html) transformer.

In [ ]:
df.columns

In [ ]:
dtfs = DatetimeFeatures(
        variables=None, features_to_extract="all",
                    )

In [ ]:
dtfs = DatetimeFeatures(
        variables=['birthdate'], 
        features_to_extract=["year"], drop_original=False
                    )

In [ ]:
# Extract features.
df2 = dtfs.fit_transform(df)

In [ ]:
df2

In [ ]:
# The datetime variable, which was automatically
# identified, is stored in an attribute.

dtfs.variables_

In [ ]:
df2["BirthDate"].value_counts().to_frame()

In [ ]:
df2["BirthDate_year"].value_counts().to_frame()

In [ ]:
#df2.to_csv("bicycleclassific.csv", index=False)

**==============================================================================================================**

## Variable Transformation

The most commonly used methods to transform variables are:

- Logarithmic transformation - $np.log(X)$
- Reciprocal transformation - $1 / X$
- Square root transformation - $sqrt(X)$
- Exponential transformation
- Box-Cox transformation
- Yeo-Johnson transformation

### Target Variable Transformation

Making our target variable normally distributed often will lead to better results

If our target is not normally distributed, we can apply a transformation to it and then fit our regression to predict the transformed values.

How can we tell if our target is normally distributed? There are two ways:

* Visually
* Using a statistical test

In [ ]:
sns.histplot(x=df.mpg, data=df);

* This is a statistical test that tests whether a distribution is normally distributed or not. It isn't perfect, but suffice it to say: 
    * This test outputs a "p-value". The _higher_ this p-value is the _closer_ the distribution is to normal.
    * Frequentist statisticians would say that you accept that the distribution is normal (more specifically: fail to reject the null hypothesis that it is normal) if p > 0.05.


In [ ]:
#p-value extremely low. Our y variable we've been dealing with this whole time was not normally distributed!
normaltest(df.price.values)

### Log transform

In [ ]:
log_price = np.log(df.price)

In [ ]:
log_price.hist();

In [ ]:
normaltest(log_price)

### Square root transformation

In [ ]:
sqrt_price = np.sqrt(df.price)

In [ ]:
sqrt_price.hist();

In [ ]:
normaltest(sqrt_price)

### Box cox

The box cox transformation is a parametrized transformation that tries to get distributions "as close to a normal distribution as possible".

It is defined as:

$$ \text{boxcox}(y_i) = \frac{y_i^{\lambda} - 1}{\lambda} $$

You can think of as a generalization of the square root function: the square root function uses the exponent of 0.5, but box cox lets its exponent vary so it can find the best one.

In [ ]:
bc_result = boxcox(df.price)

In [ ]:
boxcox_price = bc_result[0]

In [ ]:
sns.histplot(boxcox_price);

In [ ]:
normaltest(boxcox_price)

### Yeo-Johnson Transformation

**==============================================================================================================**

## String Operations

In [ ]:
df.head()

In [ ]:
#pd.to_numeric(df.Fare)

In [ ]:
#df.Fare = df.Fare.str.replace("$", "", regex = True)

In [ ]:
#df.Athlete_Name = df.Athlete_Name.str.title()

In [ ]:
#df.Athlete_Name = df.Athlete_Name.str.strip()

In [ ]:
df["name"].str.len()

In [ ]:
df["namelength"] = df["name"].apply(lambda x: len(x))

In [ ]:
df["wordcounts"] = df["videotranscriptiontext"].apply(lambda x: len(x.split()))

In [ ]:
df.head()

In [ ]:
#df.to_csv("Pokemon.csv", index=False)

**==============================================================================================================**

## Merging, Joining and Concatenating DataFrames

In [ ]:
df1 = pd.read_csv("market_1.csv")

In [ ]:
df2 = pd.read_csv("market_2.csv")

In [ ]:
df3 = pd.read_csv("market_3.csv")

### Concat dataset in Pandas

In [ ]:
train = pd.read_csv(".csv")

In [ ]:
train.head()

In [ ]:
train.tail()

In [ ]:
test = pd.read_csv("test.csv")

In [ ]:
test.head()

In [ ]:
test.tail()

In [ ]:
df = pd.concat([df1,df2,df3], axis=0, ignore_index=True)

In [ ]:
df

In [ ]:
#df.to_csv("googlefiber.csv", index=False)

### Merge in Pandas

In [ ]:
pd.merge(df1,df2,on='Product_ID')

In [ ]:
pd.merge(df1,df2,left_on='Product_name',right_on='Purchased_Product')

In [ ]:
pd.merge(df1,df2,how='inner', left_on=['Product_ID','Seller_City'], right_on=['Product_ID','City'])

In [ ]:
men0408= men2004.merge(men2008, how = "outer", on = "Athlete", suffixes= ("_2004", "_2008"), indicator= True )

In [ ]:
men2004.merge(men2008, how = "left", on = "Athlete", suffixes = ["_2004", "_2008"], indicator = True)

In [ ]:
men2004.merge(men2008, how = "right", on = "Athlete", suffixes = ["_2004", "_2008"], indicator = True)

In [ ]:
pd.merge(df1,df2,on='Product_ID',how='outer',indicator=True)

In [ ]:
pd.merge(df1,df2, on='Product_ID',how='left')

In [ ]:
pd.merge(df1,df2, on='Product_ID',how='right')

In [ ]:
#Handling Redundancy/Duplicates in Joins
pd.merge(df1.drop_duplicates(),df2,how='inner',on='Product_ID')

### Validating expected outputs

In [ ]:
pd.merge(df_s, df_p, left_on="id", right_on="student_id", validate="one_to_one")

In [ ]:
pd.merge(df_s, df_p2, left_on="id", right_on="student_id", validate="one_to_many")

### Joining on different Column Labels & Indexes

In [ ]:
men0408 = men2004.merge(men2008, how = "outer", left_on = "Name", right_on = "Athlete",
                      suffixes = ["_2004", "_2008"], indicator = True)

## Join

What about pd.join? Its a specific version of merge that works solely more on index.

In [ ]:
students2 = students.set_index("id")
contacts2 = contacts.set_index("student_id")

students2.join(contacts2)

**==============================================================================================================**

# Feature Scaling

## Data Standardization
<p>
Data is usually collected from different agencies in different formats.
(Data standardization is also a term for a particular type of data normalization where we subtract the mean and divide by the standard deviation.)
</p>

### What is standardization?

Standardisation involves centering the variable at zero, and standardising the variance to 1. The procedure involves subtracting the mean of each observation and then dividing by the standard deviation:

**z = (x - x_mean) /  std**

The result of the above transformation is **z**, which is called the z-score, and represents how many standard deviations a given observation deviates from the mean. A z-score specifies the location of the observation within a distribution (in numbers of standard deviations respect to the mean of the distribution). The sign of the z-score (+ or - ) indicates whether the observation is above (+) or below ( - ) the mean.

The shape of a standardised (or z-scored normalised) distribution will be identical to the original distribution of the variable. If the original distribution is normal, then the standardised distribution will be normal. But, if the original distribution is skewed, then the standardised distribution of the variable will also be skewed. In other words, **standardising a variable does not normalize the distribution of the data** and if this is the desired outcome, we should implement any of the techniques discussed in section 7 of the course.

In a nutshell, standardisation:

- centers the mean at 0
- scales the variance at 1
- preserves the shape of the original distribution
- the minimum and maximum values of the different variables may vary
- preserves outliers

Good for algorithms that require features centered at zero.

### Feature magnitude matters because:

- The regression coefficients of linear models are directly influenced by the scale of the variable.
- Variables with bigger magnitude / larger value range dominate over those with smaller magnitude / value range
- Gradient descent converges faster when features are on similar scales
- Feature scaling helps decrease the time to find support vectors for SVMs
- Euclidean distances are sensitive to feature magnitude.
- Some algorithms, like PCA require the features to be centered at 0.


### The machine learning models affected by the feature scale are:

- Linear and Logistic Regression
- Neural Networks
- Support Vector Machines
- KNN
- K-means clustering
- Linear Discriminant Analysis (LDA)
- Principal Component Analysis (PCA)


**Feature scaling** refers to the methods or techniques used to normalize the range of independent variables in our data, or in other words, the methods to set the feature value range within a similar scale. Feature scaling is generally the last step in the data preprocessing pipeline, performed **just before training the machine learning algorithms**.

There are several Feature Scaling techniques, which we will discuss throughout this section:

- Standardisation
- Mean normalisation
- Scaling to minimum and maximum values - MinMaxScaling
- Scaling to maximum value - MaxAbsScaling
- Scaling to quantiles and median - RobustScaling
- Normalization to vector unit length

| Name | Sklearn_class |
|-------------|------------|
|Standard scaler | Standard scaler | 
|MinMaxScaler    | MinMax Scaler   |
|MaxAbs Scaler   | MaxAbs Scaler   |
|Robust scaler   | Robust scaler   |
|Quantile Transformer_Normal | Quantile Transformer(output_distribution ='normal')|
|Quantile Transformer_Uniform| Quantile Transformer(output_distribution = 'uniform')|
|PowerTransformer-Yeo-Johnson| PowerTransformer(method = 'yeo-johnson')|
|Normalizer | Normalizer|

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
X = df.iloc[:, 0:5]
y = df.iloc[:, 5]

In [ ]:
X.values, y.values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
# standardisation: with the StandardScaler from sklearn

# set up the scaler
scaler = StandardScaler()

In [ ]:
# fit the scaler to the train set, it will learn the parameters
scaler.fit(X)

In [ ]:
X_scaled = scaler.fit_transform(X)

In [ ]:
X_scaled

In [ ]:
X_scaled.shape

In [ ]:
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
X_scaled

In [ ]:
y

In [ ]:
y.shape

In [ ]:
# fit the scaler to the train set, it will learn the parameters
scaler.fit(X_train)

In [ ]:
# transform train and test sets
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled

In [ ]:
X_test_scaled

In [ ]:
combined = np.concatenate((X_scaled, y), axis=0)
combined

In [ ]:
combined = np.concatenate((X_train_scaled, X_test_scaled), axis=0)
combined

In [ ]:
X_scaled = pd.DataFrame(combined, columns=X.columns)

In [ ]:
X_scaled

In [ ]:
X_scaled.describe()

In [ ]:
X_remain = df.iloc[:, 7:]
X_remain

In [ ]:
df2 = pd.concat([X_scaled, y],axis=1)

In [ ]:
df2

In [ ]:
#df2.to_csv("winemod.csv", index=False)

### Merging all data

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df.drop(['lotarea', 'masvnrarea', 'bsmtfinsf1', 'bsmtfinsf2', 'bsmtunfsf', 'totalbsmtsf', 
         '1stflrsf', '2ndflrsf', 'grlivarea', 'bsmtfullbath', 'bsmthalfbath', 'fullbath', 
         'halfbath', 'bedroomabvgr', 'kitchenabvgr', 'totrmsabvgrd', 'fireplaces', 
         'garagecars', 'garagearea', 'years', 'saleprice'], axis=1, inplace=True)

In [ ]:
df

In [ ]:
df3 = pd.concat([df,df2], axis=1)

In [ ]:
df3

In [ ]:
#df3.to_csv("ameshousingmod.csv", index=False)

**==============================================================================================================**

## Scaling to Minimum and Maximum values - MinMaxScaling

Minimum and maximum scaling squeezes the values between 0 and 1. It subtracts the minimum value from all the observations, and then divides it by the value range:

X_scaled = (X - X.min / (X.max - X.min)


The result of the above transformation is a distribution which values vary within the range of 0 to 1. But the mean is not centered at zero and the standard deviation varies across variables. The shape of a min-max scaled distribution will be similar to the original variable. This scaling technique is also sensitive to outliers.

In a nutshell, MinMaxScaling:

- the minimum and maximum values are 0 and 1.
- does not center the mean at 0
- variance varies across variables
- sensitive outliers

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
X = df.iloc[:, 16:21]
y = df.iloc[:, 21:]

In [ ]:
X.values, y.values

In [ ]:
# set up the scaler
minmax = MinMaxScaler()

In [ ]:
# fit the scaler to the train set, it will learn the parameters
minmax.fit(X)

In [ ]:
# transform train and test sets
X_scaled = minmax.transform(X)

In [ ]:
X_scaled[0:5]

In [ ]:
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

In [ ]:
X_scaled

In [ ]:
X_scaled.describe()

In [ ]:
df2 = pd.concat([X_scaled,y], axis=1)

In [ ]:
df2

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df.drop(['carsowned', 'children', 'totalchildren', 'income', 'age', 'buyer'], axis=1, inplace=True)

In [ ]:
df

In [ ]:
df3 = pd.concat([df,df2], axis=1)

In [ ]:
df3

In [ ]:
#df3.to_csv("bicycleclassific.csv", index=False)

**==============================================================================================================**

## Mean Normalisation


Mean normalisation involves centering the variable at zero, and re-scaling to the value range. The procedure involves subtracting the mean of each observation and then dividing by difference between the minimum and maximum value:

**x_scaled = (x - x_mean) / ( x_max - x_min)**


The result of the above transformation is a distribution that is centered at 0, and its minimum and maximum values are within the range of -1 to 1. The shape of a mean normalised distribution will be similar to the original distribution.

In a nutshell, mean normalisation:

- centers the mean at 0
- variance will be different
- the minimum and maximum values are squeezed between -1 and 1
- preserves outliers

Good for algorithms that require features centered at zero.


## Scaling to maximum value - MaxAbsScaling

Maximum absolute scaling scales the data to its absolute maximum value:

X_scaled = X / abs(X.max)

The result of the above transformation is a distribution which values vary within the range of -1 to 1. But the mean is not centered at zero and the standard deviation varies across variables.

Scikit-learn suggests that this transformer is meant for data that is centered at zero, and for sparse data.

## Scaling to quantiles and median - RobustScaling

In this procedure the median is removed from the observations and then they are scaled to the inter-quantile range (IQR). The IQR is the range between the 1st quartile (25th quantile) and the 3rd quartile (75th quantile).

X_scaled = X - X_median / ( X.quantile(0.75) - X.quantile(0.25) )

This robust scaling method produces more robust estimates for the center and range of the variable, and is recommended if the data shows outliers.

## Scaling to vector unit  length / unit norm

In this procedure we scale the components of a feature vector such that the complete vector has a length of 1 or, in other words a norm of 1. **Note** that this normalisation procedure normalises the **feature** vector, and not the **observation** vector. So we divide by the norm of the feature vector, observation per observation, across the different variables, and not by the norm of the **observation** vector, across observations for the same feature.

First, let me give you the formulas, and then I illustrate with an example.

### Scaling to unit norm, formulas

Scaling to unit norm is achieved by dividing each feature vector by either the Manhattan distance (l1 norm) or the Euclidean distance of the vector (l2 norm):

X_scaled_l1 = X / l1(X)

X_scaled_l2 = X / l2(X)


The **Manhattan distance** is given by the sum of the absolute components of the vector:

l1(X) = |x1| + |x2| + ... + |xn|


Whereas the **Euclidean distance** is given by the square root of the square sum of the component of the vector:

l2(X) = sqr( x1^2 + x2^2 + ... + xn^2 )


In the above example, x1 is variable 1, x2 variable 2, and xn variable n, and X is the data for 1 observation across variables (a row in other words).

**Note** as well that as the euclidean distance squares the values of the feature vector components, outliers have a heavier weight. With outliers, we may prefer to use l1 normalisation.


### Scaling to unit norm, examples

For example, if our data has 1 observations (1 row) and 3 variables:

- number of pets
- number of children
- age

The values for each variable for that single observation are 10, 15 and 20. Our vector X = [10, 15, 20]. Then:

l1(X) = 10 + 15 + 20 = 45

l2(X) = sqr( 10^2 + 15^2 + 20^2) = sqr( 100 + 225 + 400) = **26.9**

The euclidean distance is always smaller than the manhattan distance.


The normalised vector values are therefore:

X_scaled_l1 = [ 10/45, 15/45, 20/45 ]      =  [0.22, 0.33, 0.44]

X_scaled_l2 = [10/26.9, 15/26.9, 20/26.9 ] =  [0.37, 0.55, 0.74]


Scikit-learn recommends this scaling procedures for text classification or clustering. For example, they quote the dot product of two l2-normalized TF-IDF vectors is the cosine similarity of the vectors and is the base similarity metric for the Vector Space Model commonly used by the Information Retrieval community.

**==============================================================================================================**

## Create Train, Validation and Test Dataset

When you split the data set into three splits, what we get is the test data set. The three splits consist of training data set, validation data set and test data set. You train the model using the training data set and assess the model performance using the validation data set. You optimize the model performance using training and validation data set. Finally, you test the model generalization performance using the test data set. The test data set remains hidden during the model training and model performance evaluation stage. One can split the data into a 70:20:10 ratio. 10% of the data set can be set aside as test data for testing the model performance. 

In [ ]:
df.shape

In [ ]:
trainset = df2[0:14000]

In [ ]:
trainset

In [ ]:
testset = df2[14000:]

In [ ]:
testset

In [ ]:
trainset.to_csv("train.csv", index=False)

In [ ]:
testset.to_csv("test.csv", index=False)

In [ ]:
testset.drop(['left'], axis=1, inplace=True)

In [ ]:
testset.head()

In [ ]:
testset.to_csv("test2.csv", index=False)

**==============================================================================================================**

#### Python code done by Dennis Lam